In [ ]:
import json
import os
import signal
import subprocess
import sys
import time
from pathlib import Path
from typing import Dict, List, Optional

import optuna


# =================================================
# Local configuration
# =================================================

def find_repo_root(start: Path) -> Path:
    """Find the FraudGT repository from the notebook working directory."""
    start = start.resolve()

    for candidate in [start, *start.parents]:
        if (
            (candidate / "fraudGT").is_dir()
            and (candidate / "configs").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        f"Could not find FraudGT repository starting from: {start}"
    )


REPO_ROOT = find_repo_root(Path.cwd())

# Use the same Python environment selected as the notebook kernel.
PYTHON = Path(sys.executable)

CFG_FILE = (
    REPO_ROOT
    / "configs"
    / "BTC"
    / "full_feature"
    / "BTC-Full-GINE.yaml"
)

BASE_OUT_DIR = REPO_ROOT / "results_btc_optuna_GINE"

# Disable graphical plotting windows.
BASE_ENV = os.environ.copy()
BASE_ENV["MPLBACKEND"] = "Agg"

# How frequently to inspect validation statistics.
POLL_SECONDS = 5

# Print a warning if validation statistics take too long to appear.
NO_STATS_WARNING_SECONDS = 300


# =================================================
# Smoke-test settings
# =================================================
# Start with these small values.
# Increase them only after one trial succeeds.

N_TRIALS = 30
MAX_EPOCHS = 100
TRAIN_ITER_PER_EPOCH = 64
VAL_ITER_PER_EPOCH = 64

# Full search examples:
# N_TRIALS = 30
# MAX_EPOCHS = 100
# TRAIN_ITER_PER_EPOCH = 64
# VAL_ITER_PER_EPOCH = 64


# =================================================
# Path validation
# =================================================

if not PYTHON.exists():
    raise FileNotFoundError(f"Python executable not found: {PYTHON}")

if not CFG_FILE.exists():
    raise FileNotFoundError(f"Configuration file not found: {CFG_FILE}")

BASE_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root :", REPO_ROOT)
print("Python          :", PYTHON)
print("Configuration   :", CFG_FILE)
print("Output directory:", BASE_OUT_DIR)
print("Operating system:", os.name)


# =================================================
# Statistics helpers
# =================================================

def find_newest_val_stats_file(
    trial_out: Path,
) -> Optional[Path]:
    """
    Find the newest validation stats file under a trial directory.

    FraudGT normally creates:
        trial_x/.../val/stats.json
    """
    candidates = list(trial_out.rglob("val/stats.json"))

    if not candidates:
        return None

    return max(
        candidates,
        key=lambda path: path.stat().st_mtime,
    )


def read_stats_json_lines(
    stats_file: Path,
) -> List[Dict]:
    """
    Read FraudGT stats.json.

    FraudGT stores one JSON object per line.
    """
    records: List[Dict] = []

    if not stats_file.exists():
        return records

    with stats_file.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                # FraudGT may currently be writing this line.
                continue

            records.append(record)

    return records


def read_best_val_f1(trial_out: Path) -> float:
    """Read the highest validation F1 produced by one trial."""
    stats_file = find_newest_val_stats_file(trial_out)

    if stats_file is None:
        raise FileNotFoundError(
            f"No val/stats.json found under: {trial_out}"
        )

    stats_list = read_stats_json_lines(stats_file)

    # Keep one record per epoch.
    records_by_epoch = {}

    for record in stats_list:
        epoch = record.get("epoch")

        if epoch is not None:
            records_by_epoch[int(epoch)] = record

    valid_records = [
        record
        for record in records_by_epoch.values()
        if record.get("f1") is not None
    ]

    if not valid_records:
        raise ValueError(
            f"No valid F1 records found in: {stats_file}"
        )

    best_record = max(
        valid_records,
        key=lambda record: float(record["f1"]),
    )

    return float(best_record["f1"])


# =================================================
# Cross-platform process management
# =================================================

def stop_process_tree(process: subprocess.Popen) -> None:
    """Stop a FraudGT subprocess and its children."""
    if process.poll() is not None:
        return

    if os.name == "nt":
        # Windows
        subprocess.run(
            [
                "taskkill",
                "/PID",
                str(process.pid),
                "/T",
                "/F",
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=False,
        )
    else:
        # Linux/RunPod
        try:
            os.killpg(
                os.getpgid(process.pid),
                signal.SIGTERM,
            )
            process.wait(timeout=30)
        except Exception:
            try:
                os.killpg(
                    os.getpgid(process.pid),
                    signal.SIGKILL,
                )
            except Exception:
                process.kill()

    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()


def get_popen_options() -> Dict:
    """Return platform-specific subprocess options."""
    if os.name == "nt":
        return {
            "creationflags": subprocess.CREATE_NEW_PROCESS_GROUP,
        }

    return {
        "start_new_session": True,
    }


# =================================================
# FraudGT subprocess runner
# =================================================

def run_fraudgt_trial(
    trial: optuna.Trial,
    trial_out: Path,
    batch_size: int,
    lr: float,
    weight_decay: float,
    loss_weight: int,
) -> None:
    """
    Launch one FraudGT training process.

    Validation F1 is monitored while training runs so Optuna can
    prune underperforming trials.
    """
    trial_out.mkdir(parents=True, exist_ok=True)

    cmd = [
        str(PYTHON),
        "-m",
        "fraudGT.main",

        "--cfg",
        str(CFG_FILE),

        "--gpu",
        "0",

        "--repeat",
        "1",

        # FraudGT configuration overrides
        "out_dir",
        str(trial_out),

        "wandb.use",
        "False",

        "train.batch_size",
        str(batch_size),

        "optim.base_lr",
        str(lr),

        "optim.weight_decay",
        str(weight_decay),

        "model.loss_fun_weight",
        f"[1, {loss_weight}]",

        "optim.max_epoch",
        str(MAX_EPOCHS),

        "train.iter_per_epoch",
        str(TRAIN_ITER_PER_EPOCH),

        "val.iter_per_epoch",
        str(VAL_ITER_PER_EPOCH),

        "train.eval_period",
        "1",

        "optim.batch_accumulation",
        "1",
    ]

    log_file = trial_out / "fraudgt_subprocess.log"

    print(f"\nStarting trial {trial.number}")
    print("Command:")
    print(subprocess.list2cmdline(cmd))
    print("Log:", log_file)

    with log_file.open(
        "w",
        encoding="utf-8",
        buffering=1,
    ) as log:
        process = subprocess.Popen(
            cmd,
            cwd=str(REPO_ROOT),
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
            env=BASE_ENV,
            **get_popen_options(),
        )

        reported_epochs = set()
        start_time = time.time()
        warned_no_stats = False

        try:
            while process.poll() is None:
                stats_file = find_newest_val_stats_file(
                    trial_out
                )

                if stats_file is None:
                    elapsed = time.time() - start_time

                    if (
                        elapsed > NO_STATS_WARNING_SECONDS
                        and not warned_no_stats
                    ):
                        print(
                            f"Trial {trial.number}: no validation "
                            f"stats after "
                            f"{NO_STATS_WARNING_SECONDS} seconds. "
                            f"Training is continuing."
                        )
                        warned_no_stats = True

                    time.sleep(POLL_SECONDS)
                    continue

                records = read_stats_json_lines(stats_file)

                for record in records:
                    epoch = record.get("epoch")
                    f1 = record.get("f1")

                    if epoch is None or f1 is None:
                        continue

                    epoch = int(epoch)
                    f1 = float(f1)

                    if epoch in reported_epochs:
                        continue

                    reported_epochs.add(epoch)

                    trial.report(
                        value=f1,
                        step=epoch,
                    )

                    print(
                        f"Trial {trial.number} | "
                        f"epoch={epoch} | "
                        f"val_f1={f1:.6f}"
                    )

                    if trial.should_prune():
                        print(
                            f"Pruning trial {trial.number} at "
                            f"epoch {epoch}; val_f1={f1:.6f}"
                        )

                        stop_process_tree(process)

                        raise optuna.TrialPruned(
                            f"Pruned at epoch {epoch}"
                        )

                time.sleep(POLL_SECONDS)

            returncode = process.wait()

            if returncode != 0:
                print(f"\nTrial {trial.number} failed.")
                print(f"Read the log file:\n{log_file}")

                raise RuntimeError(
                    f"FraudGT exited with code {returncode}"
                )

        except optuna.TrialPruned:
            raise

        except KeyboardInterrupt:
            print("Notebook interrupted; stopping training.")
            stop_process_tree(process)
            raise

        except Exception:
            stop_process_tree(process)
            raise


# =================================================
# Optuna objective
# =================================================

def objective(trial: optuna.Trial) -> float:
    batch_size = trial.suggest_categorical(
        "batch_size",
        [64, 128, 256, 512, 1024, 2048],
    )

    lr = trial.suggest_float(
        "base_lr",
        1e-4,
        3e-3,
        log=True,
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        1e-4,
        log=True,
    )

    loss_weight = trial.suggest_categorical(
        "loss_weight",
        [50, 75, 100, 105, 125],
    )

    trial_out = (
        BASE_OUT_DIR
        / f"trial_{trial.number}"
    )

    trial_out.mkdir(
        parents=True,
        exist_ok=True,
    )

    run_fraudgt_trial(
        trial=trial,
        trial_out=trial_out,
        batch_size=batch_size,
        lr=lr,
        weight_decay=weight_decay,
        loss_weight=loss_weight,
    )

    val_f1 = read_best_val_f1(trial_out)

    print(
        f"\nTrial {trial.number} finished | "
        f"best_val_f1={val_f1:.6f} | "
        f"parameters={trial.params}"
    )

    return val_f1


# =================================================
# Run the Optuna study
# =================================================

sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=8,
    multivariate=True,
)

pruner = optuna.pruners.MedianPruner(
    n_startup_trials=8,
    n_warmup_steps=10,
    interval_steps=5,
)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    pruner=pruner,
    study_name="btc_GINE_pruning",
)

try:
    study.optimize(
        objective,
        n_trials=N_TRIALS,
        gc_after_trial=True,
    )
except KeyboardInterrupt:
    print("\nOptuna study interrupted by user.")


# =================================================
# Results
# =================================================

completed_trials = [
    trial
    for trial in study.trials
    if trial.state == optuna.trial.TrialState.COMPLETE
]

if completed_trials:
    print("\nBest parameters:")
    print(study.best_params)

    print("\nBest validation F1:")
    print(study.best_value)
else:
    print("\nNo trial completed successfully.")

print("\nNumber of trials:")
print(len(study.trials))

print("\nTrial summary:")

for trial in study.trials:
    print(
        f"Trial {trial.number} | "
        f"state={trial.state.name} | "
        f"value={trial.value} | "
        f"parameters={trial.params}"
    )

[I 2026-06-30 15:35:48,369] A new study created in memory with name: btc_GINE_pruning


Repository root : C:\Users\hchun\Downloads\GithubRepo\FraudGT
Python          : c:\Users\hchun\Downloads\GithubRepo\FraudGT\.venv\Scripts\python.exe
Configuration   : C:\Users\hchun\Downloads\GithubRepo\FraudGT\configs\BTC\full_feature\BTC-Full-GINE.yaml
Output directory: C:\Users\hchun\Downloads\GithubRepo\FraudGT\results_btc_optuna_GINE
Operating system: nt

Starting trial 0
Command:
c:\Users\hchun\Downloads\GithubRepo\FraudGT\.venv\Scripts\python.exe -m fraudGT.main --cfg C:\Users\hchun\Downloads\GithubRepo\FraudGT\configs\BTC\full_feature\BTC-Full-GINE.yaml --gpu 0 --repeat 1 out_dir C:\Users\hchun\Downloads\GithubRepo\FraudGT\results_btc_optuna_GINE\trial_0 wandb.use False train.batch_size 128 optim.base_lr 0.00012184186502221769 optim.weight_decay 5.39948440978744e-05 model.loss_fun_weight "[1, 105]" optim.max_epoch 100 train.iter_per_epoch 64 val.iter_per_epoch 64 train.eval_period 1 optim.batch_accumulation 1
Log: C:\Users\hchun\Downloads\GithubRepo\FraudGT\results_btc_optuna_G

[I 2026-06-30 15:44:51,275] Trial 0 finished with value: 0.6087 and parameters: {'batch_size': 128, 'base_lr': 0.00012184186502221769, 'weight_decay': 5.39948440978744e-05, 'loss_weight': 105}. Best is trial 0 with value: 0.6087.



Trial 0 finished | best_val_f1=0.608700 | parameters={'batch_size': 128, 'base_lr': 0.00012184186502221769, 'weight_decay': 5.39948440978744e-05, 'loss_weight': 105}

Starting trial 1
Command:
c:\Users\hchun\Downloads\GithubRepo\FraudGT\.venv\Scripts\python.exe -m fraudGT.main --cfg C:\Users\hchun\Downloads\GithubRepo\FraudGT\configs\BTC\full_feature\BTC-Full-GINE.yaml --gpu 0 --repeat 1 out_dir C:\Users\hchun\Downloads\GithubRepo\FraudGT\results_btc_optuna_GINE\trial_1 wandb.use False train.batch_size 1024 optim.base_lr 0.0002692655251486473 optim.weight_decay 1.6738085788752145e-05 model.loss_fun_weight "[1, 125]" optim.max_epoch 100 train.iter_per_epoch 64 val.iter_per_epoch 64 train.eval_period 1 optim.batch_accumulation 1
Log: C:\Users\hchun\Downloads\GithubRepo\FraudGT\results_btc_optuna_GINE\trial_1\fraudgt_subprocess.log
Trial 1 | epoch=0 | val_f1=0.019220
Trial 1 | epoch=1 | val_f1=0.045730
Trial 1 | epoch=2 | val_f1=0.061840
Trial 1 | epoch=3 | val_f1=0.105110
Trial 1 | epoc

[I 2026-06-30 16:07:31,361] Trial 1 finished with value: 0.43847 and parameters: {'batch_size': 1024, 'base_lr': 0.0002692655251486473, 'weight_decay': 1.6738085788752145e-05, 'loss_weight': 125}. Best is trial 0 with value: 0.6087.


...

[I 2026-07-01 02:28:21,725] Trial 28 finished with value: 0.64634 and parameters: {'batch_size': 128, 'base_lr': 0.00014846849201226043, 'weight_decay': 9.433031374795653e-06, 'loss_weight': 50}. Best is trial 8 with value: 0.67925.



Trial 28 finished | best_val_f1=0.646340 | parameters={'batch_size': 128, 'base_lr': 0.00014846849201226043, 'weight_decay': 9.433031374795653e-06, 'loss_weight': 50}

Starting trial 29
Command:
c:\Users\hchun\Downloads\GithubRepo\FraudGT\.venv\Scripts\python.exe -m fraudGT.main --cfg C:\Users\hchun\Downloads\GithubRepo\FraudGT\configs\BTC\full_feature\BTC-Full-GINE.yaml --gpu 0 --repeat 1 out_dir C:\Users\hchun\Downloads\GithubRepo\FraudGT\results_btc_optuna_GINE\trial_29 wandb.use False train.batch_size 64 optim.base_lr 0.00011027713083695568 optim.weight_decay 2.023320554744496e-05 model.loss_fun_weight "[1, 105]" optim.max_epoch 100 train.iter_per_epoch 64 val.iter_per_epoch 64 train.eval_period 1 optim.batch_accumulation 1
Log: C:\Users\hchun\Downloads\GithubRepo\FraudGT\results_btc_optuna_GINE\trial_29\fraudgt_subprocess.log
Trial 29 | epoch=0 | val_f1=0.013960
Trial 29 | epoch=1 | val_f1=0.023590
Trial 29 | epoch=2 | val_f1=0.103340
Trial 29 | epoch=3 | val_f1=0.135460
Trial 29

[I 2026-07-01 03:00:00,016] Trial 29 finished with value: 0.62222 and parameters: {'batch_size': 64, 'base_lr': 0.00011027713083695568, 'weight_decay': 2.023320554744496e-05, 'loss_weight': 105}. Best is trial 8 with value: 0.67925.



Trial 29 finished | best_val_f1=0.622220 | parameters={'batch_size': 64, 'base_lr': 0.00011027713083695568, 'weight_decay': 2.023320554744496e-05, 'loss_weight': 105}

Best parameters:
{'batch_size': 64, 'base_lr': 0.00012068823150943148, 'weight_decay': 9.17636343981185e-06, 'loss_weight': 50}

Best validation F1:
0.67925

Number of trials:
30

Trial summary:
Trial 0 | state=COMPLETE | value=0.6087 | parameters={'batch_size': 128, 'base_lr': 0.00012184186502221769, 'weight_decay': 5.39948440978744e-05, 'loss_weight': 105}
Trial 1 | state=COMPLETE | value=0.43847 | parameters={'batch_size': 1024, 'base_lr': 0.0002692655251486473, 'weight_decay': 1.6738085788752145e-05, 'loss_weight': 125}
Trial 2 | state=COMPLETE | value=0.48061 | parameters={'batch_size': 1024, 'base_lr': 0.00012476394272569445, 'weight_decay': 7.902619549708236e-05, 'loss_weight': 50}
Trial 3 | state=COMPLETE | value=0.54809 | parameters={'batch_size': 1024, 'base_lr': 0.0009519754482692679, 'weight_decay': 4.201672

: 